In [8]:
#importação bibliotecas
import os, shutil, random
from pathlib import Path

#caminhos das pastas
caminho = Path("data/dataset")
destino = Path("data/dataset_final")
random.seed(42)

#define as proporções de divisão
ratios = {'train': 0.7, 'val': 0.15, 'test': 0.15}

#recebe todas as classes
classes = [d.name for d in caminho.iterdir() if d.is_dir()]

#remove o destino se já existir e recria a estrutura
if destino.exists():
    shutil.rmtree(destino)
destino.mkdir(parents=True, exist_ok=True)

#loop por cada classe encontrada
for cls in classes:
    files = []
    #busca por imagens com extensões específicas
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff"):
        files += list((caminho/cls).glob(ext))
    
    #embaralha as imagens para aleatorizar a divisão
    files = sorted(files)
    random.shuffle(files)

    #divide proporcionalmente em train, val e test
    n = len(files)
    n_train = int(ratios['train'] * n)
    n_val   = int(ratios['val'] * n)

    splits = {
        'train': files[:n_train],
        'val': files[n_train:n_train+n_val],
        'test': files[n_train+n_val:]
    }

    #copia os arquivos para suas respectivas pastas
    for split, imgs in splits.items():
        outdir = destino / split / cls
        outdir.mkdir(parents=True, exist_ok=True)
        for img in imgs:
            shutil.copy2(img, outdir)

#exibe o total de imagens em cada conjunto
for split in ("train","val","test"):
    total = sum(1 for _ in (destino/split).rglob("*.*"))
    print(split, "->", total, "imagens")

train -> 2949 imagens
val -> 631 imagens
test -> 637 imagens


In [9]:
from pathlib import Path, PurePosixPath
import collections

#print para confirmar se está balenceado
for split in ("train","val","test"):
    classes = sorted([p.name for p in (destino/split).iterdir() if p.is_dir()])
    counts = {c: len(list((destino/split/c).glob("*.*"))) for c in classes}
    print(f"\n[{split}] classes:", classes)
    print("contagens:", counts)


[train] classes: ['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']
contagens: {'cataract': 726, 'diabetic_retinopathy': 768, 'glaucoma': 704, 'normal': 751}

[val] classes: ['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']
contagens: {'cataract': 155, 'diabetic_retinopathy': 164, 'glaucoma': 151, 'normal': 161}

[test] classes: ['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']
contagens: {'cataract': 157, 'diabetic_retinopathy': 166, 'glaucoma': 152, 'normal': 162}
